# IMPLEMENTACIÓN MÉTODO CONTRAFACTUAL CONFETTI
### Caso con predicción cercana a 0.5

Se desarrollan los códigos para lograr implementar el método CONFETTI (Cetina et al. 2026) en el modelo entrenado mediante transfer learning para predecir, a partir de 6 series de tiempo de tsunami en boyas, la probabilidad de inundación por tsunami para la localidad de Coquimbo.

Se implementa para un dato del conjunto de pruebas, con una <span style="color:magenta">predicción cercana a 0.5</span> para evaluar el comportamiento en casos intermedios y verificar que el contrafactual cumpla con el cambio de clasificación.

<u>Referencias:</u>
- Cetina, A. G. P., Benguessoum, K., Lourenço, R., & Kubler, S. (2026). Counterfactual Explainable AI (XAI) Method for Deep Learning-Based Multivariate Time Series Classification. Proceedings of the AAAI Conference on Artificial Intelligence, 17393–174000. http://arxiv.org/abs/2511.13237

In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
import numpy as np
from datetime import datetime

from scipy.interpolate import interp1d

from confetti import CONFETTI
from confetti.attribution import cam
from confetti.utils import load_multivariate_ts_from_csv
from confetti.visualizations import plot_counterfactual
import keras
from keras.models import load_model
import pickle

C:\Users\asuso\anaconda3\envs\XAI_proj_312\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
def load_pickle(p_name):
    with open(p_name, 'rb') as file:
        # Load the data from the file
        data = pickle.load(file)
    return data

data_path = 'DATA\\'
y_test = load_pickle(data_path+'ytest_new.pickle')
X_test = load_pickle(data_path+'xtest_new.pickle')
y_train = load_pickle(data_path+'ytrain_new.pickle')
X_train = load_pickle(data_path+'xtrain_new.pickle')
y_val = load_pickle(data_path+'yval_new.pickle')
X_val = load_pickle(data_path+'xval_new.pickle')

In [3]:
# OJO este modelo no es el mismo que se carga en CONFETTI!! debido a que confeti necesita que la predicción sea binaria
# Este modelo y el que se carga más adelante predicen lo mismo, solo que este entrega la salida de softmax (es decir entre 0 y 1,
# mientras que el siguiente entrega valores discretos {0, 1}.
# La instancia que se está usando es la misma para ambos

model = load_model('MODELS\\transfer_learned_tsunami_classifier3.keras')

def _prediccion(model, X, classes=['No inunda', 'Inunda'], verb=1):
    class_p = model.predict(X)[0]
    if verb==1:
        print('-----------------------------------------------------------')
        print(f'La probabilidad de inundación es {class_p[0]:.3f}. Clasifica como "{classes[np.round(class_p,0).astype(int)[0]]}"')
        print('-----------------------------------------------------------')
    return class_p

def resize_cam_weights(cam_weights, target_length):
    original_length = len(cam_weights)
    x_original = np.linspace(0, 1, original_length)
    x_target = np.linspace(0, 1, target_length)
    interpolator = interp1d(x_original, cam_weights, kind='linear')
    return interpolator(x_target)

In [4]:
try:
    training_weights=load_pickle('MODELS\\training_weights.pkl')
    print('Pesos cargados desde pickle')
except:
    print('Se calcularán los pesos y se guardaran en un archivo pickle')
    training_weights = cam(model, X_train) #ESTO SE DEMORA
    # Save to a file
    with open('MODELS\\training_weights.pkl', 'wb') as file:
        pickle.dump(training_weights, file)

Pesos cargados desde pickle


In [5]:
#Buscamos un caso con probabilidad cercana a 0.5
for i in range(len(X_test)):
    instance = X_test[i:i+1]
    p = model.predict(instance)[0]
    if (p[0] >0.4) and (p[0]<0.6):
        print(i)
        break

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 507ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 228

In [6]:
# Select instance to explain
i = 62
instance = X_test[i:i+1] #Seleccionar la serie de tiempo a evaluar

start = datetime.now()

c = _prediccion(model, instance)

end = datetime.now()
print(f'Se demoró {str(end - start)} en predecir')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
-----------------------------------------------------------
La probabilidad de inundación es 0.464. Clasifica como "No inunda"
-----------------------------------------------------------
Se demoró 0:00:00.234441 en predecir


In [7]:
@keras.saving.register_keras_serializable()
class BinaryOutputLayer(keras.layers.Layer):
    def call(self, x):
        prob_clase_1 = keras.ops.squeeze(x, axis=-1)
        prob_clase_0 = 1.0 - prob_clase_1
        return keras.ops.stack([prob_clase_0, prob_clase_1], axis=1)

explainer = CONFETTI(model_path="MODELS\\transfer_learned_tsunami_classifier4_wrapped.keras")

# results = explainer.generate_counterfactuals(
#     instances_to_explain=instance,
#     reference_data=X_train,
#     reference_weights=training_weights,
#     n_partitions=2, #Controla la cantidad de segementos temporales en que se divide la serie
#     alpha=0.9, #controla la relación confianza - sparcity
#     theta=0.51, #indica con qué valor cambia la clase
#     optimize_sparsity=False,
#     population_size=200
# )

# cam_resized = resize_cam_weights(results[0].feature_importance, 241)
# print("Shape resized:", cam_resized.shape)  # debe ser (241,)

In [9]:

def resize_cam_weights(cam_weights, target_length):
    x_orig = np.linspace(0, 1, len(cam_weights))
    x_target = np.linspace(0, 1, target_length)
    return interp1d(x_orig, cam_weights, kind='linear')(x_target)

# Redimensionar todos los pesos
training_weights_resized = np.array([
    resize_cam_weights(w, 241) for w in training_weights
])

print(f"Shape original:      {training_weights.shape}")       # (n, 72)
print(f"Shape redimensionado: {training_weights_resized.shape}")  # (n, 241)

# Verificar que el máximo ahora apunta a t≈143
#cam_nun_resized = training_weights_resized[4740]


Shape original:      (4742, 72)
Shape redimensionado: (4742, 241)


In [ ]:
# ESTO SE DEMORA, PERO PERMITE GENERAR MÁS CONTRACATUALES

# Intentar con theta más bajo y más generaciones
start = datetime.now()

results_free = explainer.generate_counterfactuals(
    instances_to_explain=instance,
    reference_data=X_train,
    reference_weights=None,
    n_partitions=5,
    population_size=300,
    maximum_number_of_generations=300,
    alpha=0.5,
    theta=0.3,          # <-- umbral más permisivo
    optimize_sparsity=True,  # <-- volvemos a activar
    verbose=True        # <-- para ver progreso
)

if results_free[0].best is not None:
    cf_free = results_free[0].best.counterfactual
    diff_free = np.abs(cf_free - instance[0])
    top5_free = np.argsort(diff_free.mean(axis=1))[::-1][:5]
    print(f"Diferencia máxima: {diff_free.max():.6f}")
    print(f"Top 5 timesteps modificados: {top5_free}")
else:
    print("No se encontró contrafactual válido")

end = datetime.now()

print(f'Se demoró {srt(end-start)}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 434ms/step
149/149 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
Skipping Naive Stage as no weights were provided.
Optimization of CE for Instance 0 started.
Optimization of CE for Instance 0 in Window 121
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
10/10 ━━━━━━━━━━━━━━

In [ ]:
cf_free = results_free[0].best.counterfactual
diff_mask = np.abs(cf_free - instance[0]).mean(axis=1) > 1e-6
timesteps_modificados = np.where(diff_mask)[0]

t_inicio = timesteps_modificados[0]
t_fin = timesteps_modificados[-1]

print(f"Zona modificada: t={t_inicio} a t={t_fin}")

fig, axes = plt.subplots(6, 1, figsize=(12, 14), sharex=True)

for i, ax in enumerate(axes):
    ax.plot(instance[0][:, i], label="Original (inunda)", color="blue")
    ax.plot(cf_free[:, i], label="Contrafactual (no inunda)", color="red", linestyle="--")
    ax.axvspan(t_inicio, t_fin, alpha=0.2, color="yellow", label=f"Zona modificada (t={t_inicio}-{t_fin})")
    ax.set_ylabel(f"Boya {i+1}")
    ax.legend(fontsize=7)

plt.xlabel("Timestep")
plt.suptitle("Original vs Contrafactual — zona crítica para clasificación")
plt.tight_layout()
plt.show()

"El modelo identifica la fase inicial de la señal (t≈13-35) como la región temporal más relevante para predecir inundación. Los contrafactuales generados por CONFETTI modifican consistentemente esta zona, sugiriendo que pequeñas diferencias en la amplitud de las primeras ondas registradas en las boyas son suficientes para cambiar la clasificación de inunda a no inunda."

In [ ]:
all_cfs = results_free[0].all_counterfactuals
print(f"Número de contrafactuales generados: {len(all_cfs)}")
print(f"Tipo: {type(all_cfs[0])}")

In [ ]:
# Calcular magnitud de modificación para cada contrafactual
cf_stats = []
for i, cf in enumerate(all_cfs):
    cf_array = cf.counterfactual
    diff = np.abs(cf_array - instance[0])
    cf_stats.append({
        "index": i,
        "cf": cf_array,
        "label": cf.label,
        "diff_max": diff.max(),
        "diff_mean": diff.mean(),
        "timesteps_modificados": int(np.sum(diff.mean(axis=1) > 1e-6)),
        "t_inicio": np.where(diff.mean(axis=1) > 1e-6)[0][0] if np.any(diff.mean(axis=1) > 1e-6) else None,
        "t_fin": np.where(diff.mean(axis=1) > 1e-6)[0][-1] if np.any(diff.mean(axis=1) > 1e-6) else None,
    })


# Ordenar por diferencia media (de menor a mayor modificación)
cf_stats_sorted = sorted(cf_stats, key=lambda x: x["diff_mean"])

# Resumen
print(f"{'#':<4} {'Diff media':<12} {'Diff máx':<12} {'Timesteps mod.':<16} {'Región'}")
print("-" * 60)
for i, s in enumerate(cf_stats_sorted):
    region = f"t={s['t_inicio']}-{s['t_fin']}" if s['t_inicio'] is not None else "N/A"
    print(f"{i:<4} {s['diff_mean']:<12.6f} {s['diff_max']:<12.6f} {s['timesteps_modificados']:<16} {region}")

In [ ]:
# Seleccionar 5 contrafactuales representativos
n = len(cf_stats_sorted)
indices_rep = [0, n//4, n//2, 3*n//4, n-1]
labels_rep = ["Mínima", "Baja", "Media", "Alta", "Máxima"]

fig, axes = plt.subplots(len(indices_rep), 1, figsize=(14, 3 * len(indices_rep)), sharex=True)

for ax, idx, lbl in zip(axes, indices_rep, labels_rep):
    s = cf_stats_sorted[idx]
    # Promedio sobre los 6 canales para simplificar
    orig_mean = instance[0].mean(axis=1)
    cf_mean = s["cf"].mean(axis=1)
    
    ax.plot(orig_mean, label="Original", color="blue")
    ax.plot(cf_mean, label=f"Contrafactual ({lbl})", color="red", linestyle="--")
    if s["t_inicio"] is not None:
        ax.axvspan(s["t_inicio"], s["t_fin"], alpha=0.2, color="yellow", label="Zona modificada")
    ax.set_title(f"Modificación {lbl} — Δmean={s['diff_mean']:.5f}, timesteps={s['timesteps_modificados']}")
    ax.legend(fontsize=8)
    ax.set_ylabel("Amplitud media")

plt.xlabel("Timestep")
plt.suptitle("Contrafactuales ordenados por magnitud de modificación", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Ordenar por t_inicio (región modificada)
cf_stats_sorted_region = sorted(
    [s for s in cf_stats if s["t_inicio"] is not None],
    key=lambda x: x["t_inicio"]
)

# Resumen
print(f"{'#':<4} {'t_inicio':<12} {'t_fin':<10} {'Timesteps mod.':<16} {'Diff media'}")
print("-" * 55)
for i, s in enumerate(cf_stats_sorted_region):
    print(f"{i:<4} {s['t_inicio']:<12} {s['t_fin']:<10} {s['timesteps_modificados']:<16} {s['diff_mean']:.6f}")

In [ ]:
# Identificar grupos de región (umbral de 10 timesteps entre grupos)
grupos = []
grupo_actual = [cf_stats_sorted_region[0]]

for s in cf_stats_sorted_region[1:]:
    if s["t_inicio"] - grupo_actual[-1]["t_inicio"] <= 10:
        grupo_actual.append(s)
    else:
        grupos.append(grupo_actual)
        grupo_actual = [s]
grupos.append(grupo_actual)

print(f"\nGrupos de región encontrados: {len(grupos)}")
for i, g in enumerate(grupos):
    t_inits = [s["t_inicio"] for s in g]
    t_fins = [s["t_fin"] for s in g]
    print(f"  Grupo {i+1}: t={min(t_inits)}-{max(t_fins)}, {len(g)} contrafactuales")

In [ ]:
#Buena observación. Si todos los contrafactuales modifican desde t≈1-17, la región de inicio es bastante consistente, y lo que realmente varía es cuántos timesteps se modifican. Reorganicemos por eso:
# Ordenar por timesteps modificados
cf_stats_sorted_ts = sorted(
    [s for s in cf_stats if s["t_inicio"] is not None],
    key=lambda x: x["timesteps_modificados"]
)

print(f"{'#':<4} {'Timesteps mod.':<16} {'t_inicio':<10} {'t_fin':<10} {'Diff media'}")
print("-" * 55)
for i, s in enumerate(cf_stats_sorted_ts):
    print(f"{i:<4} {s['timesteps_modificados']:<16} {s['t_inicio']:<10} {s['t_fin']:<10} {s['diff_mean']:.6f}")

In [ ]:
n = len(cf_stats_sorted_ts)
indices_rep = [0, n//4, n//2, 3*n//4, n-1]
labels_rep = ["Mínima extensión", "Baja", "Media", "Alta", "Máxima extensión"]

fig, axes = plt.subplots(len(indices_rep), 1, figsize=(14, 3 * len(indices_rep)), sharex=True)

for ax, idx, lbl in zip(axes, indices_rep, labels_rep):
    s = cf_stats_sorted_ts[idx]
    orig_mean = instance[0].mean(axis=1)
    cf_mean = s["cf"].mean(axis=1)

    ax.plot(orig_mean, label="Original", color="blue")
    ax.plot(cf_mean, label=f"Contrafactual ({lbl})", color="red", linestyle="--")
    ax.axvspan(s["t_inicio"], s["t_fin"], alpha=0.2, color="yellow", label=f"t={s['t_inicio']}-{s['t_fin']} ({s['timesteps_modificados']} ts)")
    ax.set_title(f"{lbl} — {s['timesteps_modificados']} timesteps modificados, Δmean={s['diff_mean']:.5f}")
    ax.legend(fontsize=8)
    ax.set_ylabel("Amplitud media")

plt.xlabel("Timestep")
plt.suptitle("Contrafactuales ordenados por extensión de modificación\n (comportamiento promedio de la onda a lo largo de las 6 boyas)", y=1.01)
plt.tight_layout()
plt.show()


Cada uno de los 6 canales corresponde a una boya que registra alturas de tsunami. El promedio que estamos graficando representa el comportamiento medio de todas las boyas simultáneamente, lo que da una visión general de la señal.
Para el propósito de comparar contrafactuales entre sí (ordenados por timesteps modificados), el promedio es una buena elección porque simplifica la visualización.

- Grupo 1 (índices 0-37): 5-15 timesteps modificados, región t≈0-55, diff media ≈ 0.0027
- Grupo 2 (índices 38-46): 36-42 timesteps modificados, región t≈0-120, diff media ≈ 0.0054

In [ ]:
# Seleccionar representativos de cada grupo
indices_rep = [0, 18, 37, 38, 46]
labels_rep = [
    "Mínima (5 ts, t=16-20)",
    "Media grupo 1 (9 ts, t=0-38)",
    "Máxima grupo 1 (15 ts, t=0-17)",
    "Mínima grupo 2 (36 ts, t=0-117)",
    "Máxima grupo 2 (42 ts, t=0-120)",
]

fig, axes = plt.subplots(len(indices_rep), 1, figsize=(14, 4 * len(indices_rep)), sharex=True)

for ax, idx, lbl in zip(axes, indices_rep, labels_rep):
    s = cf_stats_sorted_ts[idx]
    orig_mean = instance[0].mean(axis=1)
    cf_mean = s["cf"].mean(axis=1)

    ax.plot(orig_mean, label="Original (inunda)", color="blue")
    ax.plot(cf_mean, label=f"Contrafactual", color="red", linestyle="--")
    ax.axvspan(s["t_inicio"], s["t_fin"], alpha=0.2, color="yellow",
               label=f"Zona modificada: t={s['t_inicio']}-{s['t_fin']} ({s['timesteps_modificados']} ts)")
    ax.set_title(f"{lbl} — Δmean={s['diff_mean']:.5f}")
    ax.legend(fontsize=8)
    ax.set_ylabel("Amplitud media")

plt.xlabel("Timestep")
plt.suptitle("Contrafactuales representativos por extensión de modificación", y=1.01)
plt.tight_layout()
plt.show()